Libararies

In [209]:
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import os



#Loading the Optimizer Module
import sys
sys.path.append("/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject")

import teamOptimizer as to

importlib.reload(to)

<module 'teamOptimizer' from '/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject/teamOptimizer.py'>

### File Import

In [210]:
working_directory = Path.cwd().parent.parent
print(working_directory)


#FIXME: consider not having these vals hardcoded or wrapping in a read function that
#  has the ability to do both hard coded or default most recent week

file_path_drivers = working_directory / "data" / "predictions" / "drivers" / "driver_predictions_2026.csv" 
file_path_constr = working_directory / "data" / "predictions" / "constructors" / "constructor_preds_2026.csv"
file_path_roster = working_directory / "data" / "myTeam" / "teamConfig.csv"

predicted_points_constr = pd.read_csv(file_path_constr).drop(columns=["Unnamed: 0"])
predicted_points_drivers = pd.read_csv(file_path_drivers)

last_week_roster = pd.read_csv(file_path_roster).drop(columns=["Unnamed: 0"])


/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


### Filter the predicted points to the max race of the max year

In [211]:
year = max(predicted_points_drivers["year"])
race = max(predicted_points_drivers["race_num"])
print(year, race)

2026 3


In [212]:
predicted_points_constr = predicted_points_constr[(predicted_points_constr["year"] == year) & (predicted_points_constr["race_num"] == race)]
predicted_points_drivers = predicted_points_drivers[(predicted_points_drivers["year"] == year) & (predicted_points_drivers["race_num"] == race)]
#need this to be the prior week so 1- current week #FIXME: will this break when there is a 2 or three week gap between races?
last_week_roster = last_week_roster[(last_week_roster["race"] == race-1) & (last_week_roster["year"] == year)]


In [213]:
predicted_points_constr.head()

,year,race_num,constructor,price,predicted_points
22,2026,3,mer,29.9,84.876725
23,2026,3,fer,23.9,73.396170
24,2026,3,rbr,28.8,50.837738
25,2026,3,rb,7.4,38.592761
26,2026,3,mcl,28.5,33.167653


### Shrink Factor Towards the Mean

In [214]:
pred_mean = predicted_points_drivers["predicted_points"].mean()
predicted_points_drivers["predicted_points"] = (
    0.8 * predicted_points_drivers["predicted_points"] + 0.2 * pred_mean
)

In [215]:
predicted_points_drivers.sort_values("predicted_points", ascending=False).head(30)

,year,race_num,driver,price,constructor,predicted_points
44,2026,3,VER,28.1,RBR,29.829807
45,2026,3,RUS,28.0,MER,27.653004
48,2026,3,ANT,23.8,MER,23.609782
49,2026,3,LEC,23.4,FER,21.951182
50,2026,3,HAM,22.9,FER,20.611735
46,2026,3,NOR,26.8,MCL,18.886343
47,2026,3,PIA,24.9,MCL,17.443122
56,2026,3,BEA,8.6,HAS,10.625259
51,2026,3,HAD,13.9,RBR,9.771936
53,2026,3,SAI,12.2,WIL,8.604177


In [216]:
last_week_roster.head()

,Race_team,year,race,constructor_1,constructor_2,driver_1,driver_2,driver_3,driver_4,driver_5,bonus_driver,team
2,1_Guppies,2026,2,MER,RB,VER,BEA,HUL,COL,LAW,VER,Guppies
3,1_Guppies2,2026,2,MER,RB,VER,BEA,LIN,BOR,GAS,VER,Guppies2


In [217]:
last_week_lineup = {
    "drivers": {"VER", "BEA", "COL", "HUL", "LAW"}, 
    "constructors": {"MER", "RB"}
}

In [218]:
import inspect
print(inspect.signature(to.optimize_team))

(budget: float, drivers, cons, last_week_lineup: dict | None = None, free_transfers_avail: int = 2, points_col: str = 'predicted_points', n_drivers: int = 5, n_constructors: int = 2, max_drivers_per_team: int | None = None, use_drs: bool = False, drs_multiplier: float = 2.0, solver_msg: bool = False, require_driver_from_each_constructor: bool = False, min_drivers_per_selected_constructor: int = 1)


In [219]:
drivers_sel, cons_sel, summary = to.optimize_team(
    budget=103.1,
    drivers=predicted_points_drivers,
    cons=predicted_points_constr,
    last_week_lineup = last_week_lineup, 
    free_transfers_avail = 2,
    points_col="predicted_points",
    n_drivers=5,
    n_constructors=2,
    max_drivers_per_team=2,
    use_drs=True,
    drs_multiplier=2.0,
    solver_msg=False,
    require_driver_from_each_constructor=False,
    min_drivers_per_selected_constructor=1
)

print(summary)
display(drivers_sel)
display(cons_sel)

{'status': 1, 'budget': 103.1, 'points_column_used': 'predicted_points', 'total_price': 103.0, 'gross_points': 231.45, 'total_transfers': 3, 'free_transfers_avail': 2, 'paid_transfers': 1, 'transfer_penalty': 10, 'net_points': 221.45, 'drs_driver': 'LEC'}


,year,race_num,driver,price,constructor,predicted_points,team_key,driver_key
0,2026,3,LEC,23.4,FER,21.951182,FER,LEC
1,2026,3,BEA,8.6,HAS,10.625259,HAS,BEA
2,2026,3,LAW,6.9,RB,6.654415,RB,LAW
3,2026,3,BOT,4.7,CAD,6.055809,CAD,BOT
4,2026,3,HUL,5.6,AUD,5.934902,AUD,HUL


,year,race_num,constructor,price,predicted_points,team_key
0,2026,3,mer,29.9,84.876725,MER
1,2026,3,fer,23.9,73.396170,FER
